# 18 - V2 federated evaluation

Run the shared-backbone federated comparison on the versioned six-dataset protocol.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').is_dir():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Canonical RareGuard-XIDS IMPLEMENTATION project not found.')
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/gdrive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
import subprocess
import torch

DEVELOPMENT_DATASETS = [
    'cse_cic_ids2018',
    'edge_iiotset',
    'cic_bccc_nrc_tabulariotattack_2024',
    'cic_iiot_2025_datasense',
]
CONFIRMATION_DATASETS = ['ciciot2023', 'ton_iot']

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the development-dataset stage.')

def run_device_split_stage(module: str) -> None:
    for dataset in DEVELOPMENT_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cuda', '--resume'],
            check=True,
        )
    for dataset in CONFIRMATION_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cpu', '--resume'],
            check=True,
        )

run_device_split_stage('rareguard.experiments.run_publication_federated')


In [3]:
import pandas as pd
display(pd.read_csv('artifacts/publication_v2/results/federated_seed_metrics.csv'))


,dataset,seed,method,backbone,best_validation_f1_macro,rounds,clients,non_iid_alpha,communication_bytes,wall_seconds,...,f1_macro,f1_weighted,balanced_accuracy,mcc,cohen_kappa,ece_15,rare_class_count,rare_f1_macro,roc_auc_macro_ovr,pr_auc_macro
0,cse_cic_ids2018,42,RareGuard-FedAvg,RareGuard-XIDS,0.751628,5,5,0.5,15532544,5.560943,...,0.743605,0.914373,0.811237,0.900303,0.900083,0.018016,3.0,0.279089,0.992386,0.801403
1,cse_cic_ids2018,42,RareGuard-FedProx,RareGuard-XIDS,0.758631,5,5,0.5,15532544,4.790838,...,0.743174,0.914512,0.810087,0.901385,0.901136,0.016313,3.0,0.289318,0.992268,0.796630
2,cse_cic_ids2018,42,RareGuard-CFL,RareGuard-XIDS,0.742908,5,5,0.5,15532544,4.016187,...,0.714502,0.904384,0.797638,0.891013,0.890511,0.048521,3.0,0.208106,0.989240,0.791390
3,cse_cic_ids2018,42,RareGuardFed,RareGuard-XIDS,0.757597,5,5,0.5,15532544,3.490431,...,0.743695,0.913603,0.812610,0.900042,0.899814,0.017967,3.0,0.293294,0.992366,0.796816
4,cse_cic_ids2018,73,RareGuard-FedAvg,RareGuard-XIDS,0.713990,5,5,0.5,14181888,3.076358,...,0.713232,0.914310,0.789641,0.890850,0.889497,0.035230,3.0,0.169872,0.991410,0.801249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ton_iot,73,RareGuardFed,RareGuard-XIDS,0.401723,5,5,0.5,11407368,5.270108,...,0.386087,0.386087,0.420000,0.384469,0.355556,0.170132,0.0,NaN,0.876604,0.613968
68,ton_iot,101,RareGuard-FedAvg,RareGuard-XIDS,0.308828,5,5,0.5,12493784,5.817267,...,0.327269,0.327269,0.390000,0.367316,0.322222,0.112294,0.0,NaN,0.905153,0.609238
69,ton_iot,101,RareGuard-FedProx,RareGuard-XIDS,0.308828,5,5,0.5,12493784,5.872616,...,0.327269,0.327269,0.390000,0.367316,0.322222,0.112294,0.0,NaN,0.905153,0.609238
70,ton_iot,101,RareGuard-CFL,RareGuard-XIDS,0.403981,5,5,0.5,12493784,6.293922,...,0.420010,0.420010,0.477500,0.437486,0.419444,0.099333,0.0,NaN,0.935104,0.740451
